In [1]:
import bonesis
import pandas as pd
from colomoto_jupyter import tabulate

ipylab module is not installed, menus and toolbar are disabled.


BoNesis builds Boolean networks using two inputs:

1. The domain of Boolean networks

This defines which models are allowed.

It can be:

- a single Boolean network (for verification or reprogramming), or

- an influence graph, which defines: the components (genes), who influences whom and whether the influence is activating or inhibiting.

Let us define an influence graph from a list of pairwise interactions, with a sign.

In [2]:
influences = [
("Pax6","Pax6",dict(sign=1)),
("Pax6","Hes5",dict(sign=1)),
("Pax6","Mash1",dict(sign=1)),
("Hes5","Mash1",dict(sign=-1)),
("Hes5","Scl",dict(sign=1)),
("Hes5","Olig2",dict(sign=1)),
("Hes5","Stat3",dict(sign=1)),
("Mash1","Hes5",dict(sign=-1)),
("Mash1","Zic1",dict(sign=1)),
("Mash1","Brn2",dict(sign=1)),
("Zic1","Tuj1",dict(sign=1)),
("Brn2","Tuj1",dict(sign=1)),
("Scl","Olig2",dict(sign=-1)),
("Scl","Stat3",dict(sign=1)),
("Olig2","Scl",dict(sign=-1)),
("Olig2","Myt1L",dict(sign=1)),
("Olig2","Sox8",dict(sign=1)),
("Olig2","Brn2",dict(sign=-1)),
("Stat3","Aldh1L1",dict(sign=1)),
("Myt1L","Tuj1",dict(sign=1)),
]

In [3]:
dom1 = bonesis.InfluenceGraph(influences, allow_skipping_nodes= True)
dom1

# computing graph layout...


Here, dom1 delimits any BN that uses at most the listed influences, with the right sign. Thus, some solutions may use only a subset of this influence graph.

dom1 = a domain of candidate Boolean networks.

“All Boolean networks that:

- have these genes,
- respect these regulatory interactions,
- and do not violate the sign of regulation.”

In [4]:
print(f"domain: {len(dom1.nodes())} nodes, {len(dom1.edges())} edges")

domain: 12 nodes, 20 edges


If you want to enforce BNs using all the given influences, use the option exact=True:

In [5]:
dom2 = bonesis.InfluenceGraph(influences, allow_skipping_nodes= True, exact=True)

For influence graphs with large in-degrees, it is necessary to specify a bound on the number of clauses in the disjunction normal form (DNF) of the BNs with the maxclause argument. See help(bonesis.InfluenceGraph) for other options.

Why all of these boolean models are allowed? Because the influence graph only says: (e.g.)
- Pax6 → Mash1 is positive
- Hes5 → Mash1 is negative

It does not say:

- how strong these effects are,
- whether both are required,
- whether one can override the other,
- whether extra conditions are needed.

So BoNesis allows all Boolean functions consistent with the signs.

2. Observations of system behavior

This is usually a table or dictionary,

- describing partial observations of gene activity,

- often coming from experiments (e.g. RNA-seq).

BoNesis will later enforce that the synthesized networks are consistent with these observations.

In BoNesis, observations are constraints on the behavior of the system, not on the wiring.

They are specified by a Python dictionnary associating observation names to observed states of a subset of nodes:

Observations constrain dynamics, not structure

Influence graph → restricts structure
Observations → restrict long-term behavior

BoNesis keeps only Boolean networks that:

- respect the influence graph,
- AND can produce the observed behaviors under MP dynamics.

Why observations kill the combinatorial explosion

Recall: influence graph → millions of models.

Each observation:

- removes models that cannot generate it.
- After a few well-chosen observations only a tiny fraction of models survive.

Observations tell BoNesis which long-term behaviors must (or must not) exist, and any Boolean network that cannot reproduce them is thrown away.

In [6]:
data = {
    "zero": {n: 0 for n in dom1}, # all nodes are 0
    "init": {n: 1 if n == "Pax6" else 0 for n in dom1}, # all nodes are 0 but Pax6
    #### PARTIAL STATES
    "tM": {"Pax6": 1, "Tuj1": 0, "Scl": 0, "Aldh1L1": 0, "Olig2": 0, "Sox8": 0},
    "fT": {"Pax6": 1, "Tuj1": 1, "Brn2": 1, "Zic1": 1, "Aldh1L1": 0, "Sox8": 0},
    "tO": {"Pax6": 1, "Tuj1": 0 ,"Scl": 0, "Aldh1L1": 0, "Olig2": 1, "Sox8": 0},    
    "fMS": {"Pax6": 1, "Tuj1": 0, "Zic1": 0, "Brn2": 0, "Aldh1L1": 0, "Sox8": 1},
    "tS": {"Pax6": 1, "Tuj1": 0, "Scl": 1, "Aldh1L1": 0, "Olig2": 0, "Sox8": 0},
    "fA": {"Pax6": 1, "Tuj1": 0, "Zic1": 0, "Brn2": 0, "Aldh1L1": 1, "Sox8": 0},
}
pd.DataFrame.from_dict(data, orient="index").fillna('')

,Pax6,Hes5,Mash1,Scl,Olig2,Stat3,Zic1,Brn2,Tuj1,Myt1L,Sox8,Aldh1L1
zero,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0,0
init,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0,0
tM,1,,,0.0,0.0,,,,0,,0,0
fT,1,,,,,,1.0,1.0,1,,0,0
tO,1,,,0.0,1.0,,,,0,,0,0
fMS,1,,,,,,0.0,0.0,0,,1,0
tS,1,,,1.0,0.0,,,,0,,0,0
fA,1,,,,,,0.0,0.0,0,,0,1


What partial observations mean?

A partial observation means: “In this condition, we know the values of some genes, but we don’t care (or don’t know) about the others.”

BoNesis will accept any complete state that:

- matches these specified values,
- regardless of unspecified genes.

## Dynamical properties

This line ties everything together:

dom1
→ the domain of possible Boolean networks (from the influence graph)

data
→ the observations (partial gene-expression patterns)

Now bo is an object that:

- knows which Boolean networks are allowed,
- knows which observations exist,
- lets you express dynamical properties involving those observations.

Think of bo as:

“The problem instance: search for Boolean networks consistent with this structure and these observations.”

In [20]:
bo = bonesis.BoNesis(dom1, data)

In [21]:
bo.settings["parallel"] = 10

The data dictionnary specifies observations that can be used to constraint configurations (or states) of the network.

There are two shortcuts for binding a configuration to an observation: ~bo.obs("A") is a unique pre-defined configuration bound to the observation "A"; +bo.obs("A") returns a new configuration bound to "A". Thus in the following code:

In [22]:
cfg1 = ~bo.obs("tM")
cfg2 = ~bo.obs("tM")
cfg3 = +bo.obs("tM")
cfg4 = +bo.obs("tM")

cfg1 and cfg2 refers to the same configuration; whereas cfg3 and cfg4 may be different.

~bo.obs("A") means “the same state matching A”, while +bo.obs("A") means “some state matching A”.

In [23]:
cfg1
cfg3

<bonesis.language.ManagedIface.__init__.<locals>.managed.<locals>.Managed at 0x72ba87365a80>

## Attractor constraints

We detail two kind of attractor constraints: fixed points and trap spaces. Both are specified with the fixed predicate, which, depending on the argument will enforce the existence of one of the two kinds of attractor.



### Fixed points

When giving a configuration as argument, fixed ensures that the configuration is a fixed point:

In [25]:
bo.fixed(~bo.obs("fA"))
bo.fixed(~bo.obs("fMS"));

What this means 

For each line:

“There exists a fixed point of the Boolean network that matches observation fA (or fMS).”

A fixed point means:

- every gene keeps the same value forever,
- no update (of any kind) can change the state.

Why ~bo.obs("fA")? 

- ~ means “THE configuration matching fA”
- all constraints involving fA refer to the same exact state

So BoNesis enforces:

- one concrete configuration,
- consistent with the values in fA,
- that is a fixed point of the network.

### Trap spaces

In [26]:
fT_tp = bo.fixed(bo.obs("fT"))

A trap space is: A set of states where some genes are frozen, but others may still vary — and once the system enters this set, it can never leave.

Think of it as:

- “these genes are locked”
- “the rest don’t matter anymore”

Under MP dynamics, attractors = minimal trap spaces, so this is a very natural object.

Why bo.obs("fT") without ~?

- refers to an observation, not a single configuration

- represents all configurations compatible with fT

Fixed points pin down an exact state; trap spaces pin down only the genes that must stay fixed. BoNesis uses both to express precise or flexible biological knowledge under Most Permissive dynamics.

### Reachability constraints

Reachability answers the question:

“Starting from state A, is it possible for the system to ever end up in state (or attractor) B?”

This is not about how fast or how often — just whether at least one valid trajectory exists under Most Permissive (MP) dynamics.

In [27]:
~bo.obs("init") >= ~bo.obs("tM") >= fT_tp
~bo.obs("init") >= ~bo.obs("tO") >= ~bo.obs("fMS")
~bo.obs("init") >= ~bo.obs("tS") >= ~bo.obs("fA");

This is read left to right:

Starting from the configuration matching init, the system can reach a configuration matching tM, and from there it can reach the trap space fT_tp.

Important:

- this does not mean every path does this,
- only that one valid path exists.

### Absence of a trajectory

There exists no trajectory from cfg1 to cfg2.

In [28]:
~bo.obs("zero") / fT_tp

nonreach('<bonesis.language.ManagedIface.__init__.<locals>.managed.<locals>.Managed object at 0x72ba86b39930>', '<bonesis.language.ManagedIface.__init__.<locals>.managed.<locals>.Managed object at 0x72ba86b38f40>')

Starting from the all-zero state,
it is impossible to ever reach the trap space fT.

So the system cannot spontaneously end up in that fate from zero.

In [29]:
~bo.obs("zero") / ~bo.obs("fMS")
~bo.obs("zero") / ~bo.obs("fA")

nonreach('<bonesis.language.ManagedIface.__init__.<locals>.managed.<locals>.Managed object at 0x72ba86b39000>', '<bonesis.language.ManagedIface.__init__.<locals>.managed.<locals>.Managed object at 0x72ba86b399c0>')

From the zero state, it is impossible to reach the fixed point fMS
and impossible to reach the fixed point fA.

### Optimizations

In [30]:
bo.maximize_nodes()
#bo.maximize_constants()
bo.maximize_strong_constants()

<bonesis.language.ManagedIface.__init__.<locals>.managed.<locals>.Managed at 0x72ba8702c850>

### View nodes

In [31]:
#view = bonesis.NonConstantNodesView(bo, mode="opt")
view = bonesis.NonStrongConstantNodesView(bo, mode="opt")
#view = bonesis.NodesView(bo, mode="opt")
solution = next(iter(view))
for node in solution:
    print(node)

Grounding...done in 0.1s
Zic1
Tuj1
Olig2
Pax6
Mash1
Sox8
Aldh1L1
Brn2
Scl
Stat3
Hes5


In [32]:
view.standalone(output_filename="NonStrongConstantNodesView.sh")

Inference of diverse subset of solution

In [1]:
solutions = []
for bn in bo.boolean_networks(): 
    if len(solutions) > 1000:
        break
    solutions.append(bn)
    print(len(solutions))
print(len(solutions))

NameError: name 'bo' is not defined

### Enumeration of compatible BNs

Sampling with diversity

In [ ]:
bo.diverse_boolean_networks()

Enumerations of solutions are done through iterators. The basic one being the boolean_networks which returns mpbn.MPBooleanNetwork objects.

In [38]:
for bn in bo.boolean_networks(limit = 1000): # limit is optional
    print(bn)

Grounding...done in 0.1s
Aldh1L1 <- Stat3
Brn2 <- Mash1
Hes5 <- !Mash1
Mash1 <- !Hes5
Myt1L <- 0
Olig2 <- !Scl&Hes5
Pax6 <- Pax6
Scl <- !Olig2
Sox8 <- Olig2
Stat3 <- Scl&Hes5
Tuj1 <- Zic1
Zic1 <- Mash1

Aldh1L1 <- Stat3
Brn2 <- Mash1
Hes5 <- !Mash1
Mash1 <- !Hes5
Myt1L <- 0
Olig2 <- !Scl&Hes5
Pax6 <- Pax6
Scl <- !Olig2
Sox8 <- Olig2
Stat3 <- Scl&Hes5
Tuj1 <- Zic1
Zic1 <- Mash1

Aldh1L1 <- Stat3
Brn2 <- Mash1
Hes5 <- !Mash1
Mash1 <- !Hes5
Myt1L <- 0
Olig2 <- !Scl&Hes5
Pax6 <- Pax6
Scl <- !Olig2
Sox8 <- Olig2
Stat3 <- Scl&Hes5
Tuj1 <- Zic1
Zic1 <- Mash1

Aldh1L1 <- Stat3
Brn2 <- Mash1
Hes5 <- !Mash1
Mash1 <- !Hes5
Myt1L <- 0
Olig2 <- !Scl&Hes5
Pax6 <- Pax6
Scl <- !Olig2
Sox8 <- Olig2
Stat3 <- Scl&Hes5
Tuj1 <- Zic1
Zic1 <- Mash1

Aldh1L1 <- Stat3
Brn2 <- Mash1
Hes5 <- !Mash1
Mash1 <- !Hes5
Myt1L <- 0
Olig2 <- !Scl&Hes5
Pax6 <- Pax6
Scl <- !Olig2
Sox8 <- Olig2
Stat3 <- Scl&Hes5
Tuj1 <- Zic1
Zic1 <- Mash1

Aldh1L1 <- Stat3
Brn2 <- Mash1
Hes5 <- !Mash1
Mash1 <- !Hes5
Myt1L <- 0
Olig2 <- !Scl

View as data frame

In [37]:
solutions = list(bo.boolean_networks(limit = 1000))
pd.DataFrame(solutions)

Grounding...done in 0.1s


,Aldh1L1,Brn2,Hes5,Mash1,Myt1L,Olig2,Pax6,Scl,Sox8,Stat3,Tuj1,Zic1
0,Stat3,Mash1,Pax6&!Mash1,!Hes5&Pax6,0,!Scl&Hes5,Pax6,!Olig2&Hes5,Olig2,Scl&Hes5,Brn2,Mash1
1,Stat3,Mash1,Pax6&!Mash1,!Hes5&Pax6,0,!Scl&Hes5,Pax6,!Olig2&Hes5,Olig2,Scl&Hes5,Brn2,Mash1
2,Stat3,Mash1,Pax6&!Mash1,!Hes5&Pax6,0,!Scl&Hes5,Pax6,!Olig2&Hes5,Olig2,Scl&Hes5,Brn2,Mash1
3,Stat3,Mash1,Pax6&!Mash1,!Hes5&Pax6,0,!Scl&Hes5,Pax6,!Olig2&Hes5,Olig2,Scl&Hes5,Brn2,Mash1
4,Stat3,Mash1,Pax6&!Mash1,!Hes5&Pax6,0,!Scl&Hes5,Pax6,!Olig2&Hes5,Olig2,Scl&Hes5,Brn2,Mash1
...,...,...,...,...,...,...,...,...,...,...,...,...
995,Stat3,Mash1,Pax6&!Mash1,!Hes5&Pax6,0,!Scl,Pax6,!Olig2,Olig2,Scl&Hes5,Zic1|(Myt1L&Brn2),Mash1
996,Stat3,Mash1,Pax6&!Mash1,!Hes5&Pax6,0,!Scl,Pax6,!Olig2,Olig2,Scl&Hes5,Zic1|(Myt1L&Brn2),Mash1
997,Stat3,Mash1,Pax6&!Mash1,!Hes5&Pax6,0,!Scl,Pax6,!Olig2,Olig2,Scl&Hes5,Zic1|(Myt1L&Brn2),Mash1
998,Stat3,Mash1,Pax6&!Mash1,!Hes5,0,!Scl&Hes5,Pax6,!Olig2&Hes5,Olig2,Scl&Hes5,Myt1L|Zic1,Mash1


### Exportation

In [39]:
view = bo.boolean_networks()
view.standalone(output_filename="tutorial.asp")